# 02 — Define H→WW Observable and Likelihood

Implements the asymmetric Gaussian likelihood for your boosted VBF H→WW signal-strength measurement.

In [ ]:
from pathlib import Path
import json, sys, os, math
import numpy as np
import pandas as pd

def find_repo_root(start=None, marker='smeft_contents.txt', max_up=8):
    p = Path(start or Path.cwd()).resolve()
    for _ in range(max_up+1):
        if (p / marker).exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError(f'Could not find repo root containing {marker} from {Path.cwd()}')

REPO = find_repo_root()
NOTEBOOK_DIR = REPO / 'notebooks_hww_fit'
SMEFT_LISTING = REPO / 'smeft_contents.txt'
LOCAL_DICT = NOTEBOOK_DIR / 'dictionary.json'
print('repo:', REPO)
MU_OBS=0.45
SIGMA_UP=0.96
SIGMA_DN=0.78
print('Using measurement: mu = 0.45 +0.96 -0.78')

In [ ]:
def chi2_asym(mu_pred, mu_obs=MU_OBS, sigma_up=SIGMA_UP, sigma_dn=SIGMA_DN):
    d = mu_pred - mu_obs
    sigma = sigma_up if d >= 0 else sigma_dn
    return (d/sigma)**2

def nll_asym(mu_pred):
    return 0.5*chi2_asym(mu_pred)

for m in [0.0,0.45,1.0,1.5]:
    print(m, chi2_asym(m))

In [ ]:
def mu_from_xsec(sig_pred, sig_sm):
    if sig_sm <= 0:
        raise ValueError('sig_sm must be positive')
    return sig_pred / sig_sm

def xsec_from_mu(mu, sig_sm):
    return mu * sig_sm

In [ ]:
# Save measurement config for reuse
config={
 'channel':'HWW_boosted_VBF',
 'observable':'signal_strength',
 'mu_obs':MU_OBS,
 'sigma_up':SIGMA_UP,
 'sigma_dn':SIGMA_DN,
 'selection':'m_jj > 1 TeV (user-provided)'
}
(NOTEBOOK_DIR/'hww_measurement.json').write_text(json.dumps(config, indent=2))
print('wrote hww_measurement.json')